In [1]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import pyspark.sql.functions as F

# Initialize Spark session
spark = SparkSession.builder \
    .appName("confirmation") \
    .getOrCreate()  


data = [[3, '2020-03-21 10:16:13'], [7, '2020-01-04 13:57:59'], [2, '2020-07-29 23:09:44'], [6, '2020-12-09 10:39:37']]
signups = pd.DataFrame(data, columns=['user_id', 'time_stamp']).astype({'user_id':'Int64', 'time_stamp':'datetime64[ns]'})
data = [[3, '2021-01-06 03:30:46', 'timeout'], [3, '2021-07-14 14:00:00', 'timeout'], [7, '2021-06-12 11:57:29', 'confirmed'], [7, '2021-06-13 12:58:28', 'confirmed'], [7, '2021-06-14 13:59:27', 'confirmed'], [2, '2021-01-22 00:00:00', 'confirmed'], [2, '2021-02-28 23:59:59', 'timeout']]
confirmations = pd.DataFrame(data, columns=['user_id', 'time_stamp', 'action']).astype({'user_id':'Int64', 'time_stamp':'datetime64[ns]', 'action':'object'})

signups_spark = spark.createDataFrame(signups)
confirmations_spark = spark.createDataFrame(confirmations)  

print("Signups DataFrame :")
signups_spark.show()
print("Confirmations DataFrame :")
confirmations_spark.show()

'''
confirmations_count = confirmations_spark\
    .groupby('user_id')\
    .agg(F.count(F.when(col('action') == 'confirmed', True)).alias('confirmed_count').cast('decimal(3,2)'))\
    .agg(F.count('user_id').alias('total_count').cast('decimal(3,2)')

final_df2 = signups_spark.join(confirmations_count, on='user_id', how='left')\
    .select('user_id', 
            when(col('confirmed_count').isnull | col('confirmed_count') == 0, 0)\
            .when((col(confirmed_count') >= 0), col('confirmed_count'/col('total_count'))))
                       
'''
# Compute confirmed and total counts per user

confirmations_count = (
    confirmations_spark
    .groupBy("user_id")
    .agg(
        F.count("*").alias("total_count"),
        F.sum(F.when(F.col("action") == "confirmed", 1).otherwise(0)).alias("confirmed_count")
    )
)

# Join with signups and compute ratio
final_df2 = (
    signups_spark
    .join(confirmations_count, on="user_id", how="left")
    .select(
        "user_id",
        F.when(F.col("confirmed_count").isNull() | (F.col("confirmed_count") == 0), F.lit(0))
         .otherwise(F.col("confirmed_count") / F.col("total_count"))
         .alias("confirmation_ratio")
    )
)

print("Final DataFrame :")
final_df2.show()


25/12/25 13:53:10 WARN Utils: Your hostname, Yashwanths-Mac-mini.local resolves to a loopback address: 127.0.0.1; using 192.168.1.2 instead (on interface en1)
25/12/25 13:53:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/25 13:53:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Signups DataFrame :


+-------+-------------------+
|user_id|         time_stamp|
+-------+-------------------+
|      3|2020-03-21 10:16:13|
|      7|2020-01-04 13:57:59|
|      2|2020-07-29 23:09:44|
|      6|2020-12-09 10:39:37|
+-------+-------------------+

Confirmations DataFrame :
+-------+-------------------+---------+
|user_id|         time_stamp|   action|
+-------+-------------------+---------+
|      3|2021-01-06 03:30:46|  timeout|
|      3|2021-07-14 14:00:00|  timeout|
|      7|2021-06-12 11:57:29|confirmed|
|      7|2021-06-13 12:58:28|confirmed|
|      7|2021-06-14 13:59:27|confirmed|
|      2|2021-01-22 00:00:00|confirmed|
|      2|2021-02-28 23:59:59|  timeout|
+-------+-------------------+---------+

Final DataFrame :
+-------+------------------+
|user_id|confirmation_ratio|
+-------+------------------+
|      3|               0.0|
|      7|               1.0|
|      2|               0.5|
|      6|               0.0|
+-------+------------------+

